# Inférence statistique

## Premiers tests de modélisation

In [21]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [22]:
df_train = pd.read_csv("../data_finale/featuring/train_featured.csv")
df_val = pd.read_csv("../data_finale/featuring/val_featured.csv")
df_test = pd.read_csv("../data_finale/featuring/test_featured.csv")

In [23]:
colonne_cible = "market_value_in_eur"
colonne_joueur = "player"
colonne_team = "team"
colonne_nation = "nation"

df_train = df_train.dropna(subset=[colonne_cible])
df_val = df_val.dropna(subset=[colonne_cible])
df_test = df_test.dropna(subset=[colonne_cible])

# Séparation des features et de la variable cible
joueurs_test = df_test[colonne_joueur]

# On supprime les colonnes texte et la cible pour l'entraînement
X_train = df_train.drop(columns=[colonne_cible, colonne_joueur, colonne_team, colonne_nation, "position",
                                 "market_value_in_eur_nor"])
y_train = df_train[colonne_cible]

X_val = df_val.drop(columns=[colonne_cible, colonne_joueur, colonne_team, colonne_nation, "position",
                                 "market_value_in_eur_nor"])
y_val = df_val[colonne_cible]

X_test = df_test.drop(columns=[colonne_cible, colonne_joueur, colonne_team, colonne_nation, "position",
                                 "market_value_in_eur_nor"])
y_test = df_test[colonne_cible]

# Définition des modèles
modeles = {
    "Random Forest": RandomForestRegressor(
        random_state=1308,
        n_jobs=-1,
        n_estimators=200,      # Nombre d'arbres
        max_depth=30,          # Profondeur max
        min_samples_split=5    # Seuil de division
    ),
    "XGBoost": XGBRegressor(
        random_state=1308,
        n_jobs=-1,
        n_estimators=300,
        learning_rate=0.05,    # Taux d'apprentissage
        max_depth=6
    ),
}

In [24]:
# Entraînement puis évaluation
resultats = {}

for nom, modele in modeles.items():
    print(f"Entraînement de {nom}...")
    
    # Entraînement sur le jeu de train uniquement
    modele.fit(X_train, y_train)
    
    # Prédictions
    preds_val = modele.predict(X_val)
    
    mae_val = mean_absolute_error(y_val, preds_val)
    r2_val = r2_score(y_val, preds_val)
    
    # Sauvegarde pour le tableau final
    resultats[nom] = {
        "MAE Val": mae_val,
        "R² Val": r2_val,
    }
    
    print(f"   -> Validation | Erreur moyenne : {mae_val} € | Score R² : {r2_val:.2%}")

# Comparaion des modèles
print("CLASSEMENT FINAL (Trié par la plus petite erreur sur test)")
tableau_resultats = []
for nom, metrics in resultats.items():
    tableau_resultats.append({
        "Modèle": nom,
        "Erreur moyenne Val (MAE)": f"{metrics['MAE Val']} €",
        "Score R² Val": f"{metrics['R² Val']:.2%}"
    })

# En régression, le meilleur modèle est celui avec la MAE la plus basse
df_final = pd.DataFrame(tableau_resultats).sort_values(by="Erreur moyenne Val (MAE)", ascending=True)
print(df_final.to_string(index=False))

Entraînement de Random Forest...
   -> Validation | Erreur moyenne : 5050056.178193945 € | Score R² : 70.72%
Entraînement de XGBoost...
   -> Validation | Erreur moyenne : 4771397.77723274 € | Score R² : 74.64%
CLASSEMENT FINAL (Trié par la plus petite erreur sur test)
       Modèle Erreur moyenne Val (MAE) Score R² Val
      XGBoost       4771397.77723274 €       74.64%
Random Forest      5050056.178193945 €       70.72%


In [25]:
# Entraînement puis évaluation
resultats = {}

for nom, modele in modeles.items():
    print(f"Entraînement de {nom}...")
    
    # Entraînement sur le jeu de train uniquement
    modele.fit(X_train, y_train)
    
    # Prédictions
    preds_test = modele.predict(X_test)
    
    mae_test = mean_absolute_error(y_test, preds_test)
    r2_test = r2_score(y_test, preds_test)
    
    # Sauvegarde pour le tableau final
    resultats[nom] = {
        "MAE Test": mae_test,
        "R² Test": r2_test
    }
    
    print(f"   -> Test Final | Erreur moyenne : {mae_test:,.0f} € | Score R² : {r2_test:.2%}\n")

# Comparaion des modèles
print("CLASSEMENT FINAL (Trié par la plus petite erreur sur test)")
tableau_resultats = []
for nom, metrics in resultats.items():
    tableau_resultats.append({
        "Modèle": nom,
        "Erreur moyenne Test (MAE)": f"{metrics['MAE Test']:,.0f} €",
        "Score R² Test": f"{metrics['R² Test']:.2%}"
    })

# En régression, le meilleur modèle est celui avec la MAE la plus basse
df_final = pd.DataFrame(tableau_resultats).sort_values(by="Erreur moyenne Test (MAE)", ascending=True)
print(df_final.to_string(index=False))

Entraînement de Random Forest...
   -> Test Final | Erreur moyenne : 5,371,494 € | Score R² : 67.78%

Entraînement de XGBoost...
   -> Test Final | Erreur moyenne : 5,118,851 € | Score R² : 71.71%

CLASSEMENT FINAL (Trié par la plus petite erreur sur test)
       Modèle Erreur moyenne Test (MAE) Score R² Test
      XGBoost               5,118,851 €        71.71%
Random Forest               5,371,494 €        67.78%


## Auto-ML

PROBLEME : pycaret ne supporte pas encore officiellement Python 3.14.3

In [20]:
!pip install pycaret

^C


In [ ]:
import pandas as pd
import numpy as np

# Si tu utilises PyCaret v4 (recommandé)
from pycaret.regression import RegressionExperiment

# Si tu utilises PyCaret v3
# from pycaret.regression import *

print("Environnement configuré avec succès !")

ModuleNotFoundError: No module named 'pycaret'